### Step-by-step minimal Apache Beam Program

In [1]:
import apache_beam as beam
import numpy as np
import os
import csv
from apache_beam.options.pipeline_options import PipelineOptions

# Beam DoFn to load .npy files and return metadata
class LoadNpyFile(beam.DoFn):
    def process(self, file_path):
        try:
            arr = np.load(file_path)
            # Output: CSV-like string (no need to write actual CSV here)
            yield f"{os.path.basename(file_path)},{arr.shape},{arr.dtype}"
        except Exception as e:
            # Send errors to a tagged output
            yield beam.pvalue.TaggedOutput("errors", f"{file_path},ERROR,{str(e)}")

def run():
    # Adjust this to your actual folder
    input_dir = os.path.expanduser(
        "/Users/yuandouwang/Documents/projects/data_augmentation/6g_open_data_preprocessing/6g_open_data_from_KUL/nomadic_dataset/DIS_lab_LoS"
    )

    # Output CSV file name
    output_csv_file = "npy_metadata_output"
    output_error_file = "npy_load_errors.txt"

    # Gather .npy file paths
    npy_files = [
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".npy")
    ]

    # Avoid command-line argument parsing warnings
    pipeline_options = PipelineOptions([])

    with beam.Pipeline(options=pipeline_options) as p:
        results = (
            p
            | "Create file list" >> beam.Create(npy_files)
            | "Load .npy files" >> beam.ParDo(LoadNpyFile()).with_outputs("errors", main="successes")
        )

        # Add header and write successes to CSV
        (
            results.successes
            | "Add CSV header" >> beam.CombineGlobally(
                lambda lines: ["filename,shape,dtype"] + list(lines) if lines else []
            ).without_defaults()
            | "Write CSV" >> beam.io.WriteToText(
                output_csv_file,
                file_name_suffix=".csv",
                shard_name_template=""
            )
        )

        # Write error messages if any
        (
            results.errors
            | "Write errors to txt" >> beam.io.WriteToText(
                output_error_file,
                file_name_suffix="",
                shard_name_template=""
            )
        )

if __name__ == "__main__":
    run()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/yuandouwang/Documents/projects/data_augmentation/6g_open_data_preprocessing/6g_open_data_from_KUL/nomadic_dataset/DIS_lab_LoS'

#### Load .npy file and print the DataFrame.

In [ ]:
import os
import numpy as np
import pandas as pd

# Adjust this to your actual folder
input_dir = os.path.expanduser(
        "/Users/yuandouwang/Documents/projects/data_augmentation/6g_open_data_preprocessing/6g_open_data_from_KUL/ultra_dense/ULA_lab_LoS"
    )

# List all .npy files
npy_files = [
    os.path.join(input_dir, f)
    for f in os.listdir(input_dir)
    if f.endswith(".npy")
]
print(f"Found {len(npy_files)} .npy files in {input_dir}")

# Loop over files and print contents
for path in npy_files:
    try:
        arr = np.load(path)
        print(f"\n=== {os.path.basename(path)} ===\n Shape: {arr.shape}, Dtype: {arr.dtype}")
        if arr.ndim == 2:
            df = pd.DataFrame(arr)
            print(df)
        elif arr.ndim == 1:
            print("1D array:", arr)
        else:
            print(f"Array with shape {arr.shape} (not displayed)")
    except Exception as e:
        print(f"Error loading {path}: {e}")

Found 2 .npy files in /Users/yuandouwang/Documents/projects/data_augmentation/autofeat/6g_open_data_preprocessing/6g_open_data_from_KUL/ultra_dense/ULA_lab_LoS

=== antenna_positions.npy ===
 Shape: (64, 3), Dtype: float64
         0    1       2
0   2205.0  0.0  1000.0
1   2135.0  0.0  1000.0
2   2065.0  0.0  1000.0
3   1995.0  0.0  1000.0
4   1925.0  0.0  1000.0
..     ...  ...     ...
59 -1925.0  0.0  1000.0
60 -1995.0  0.0  1000.0
61 -2065.0  0.0  1000.0
62 -2135.0  0.0  1000.0
63 -2205.0  0.0  1000.0

[64 rows x 3 columns]

=== user_positions.npy ===
 Shape: (252004, 3), Dtype: int64
           0     1    2
0      -1437  1155  400
1      -1437  1160  400
2      -1437  1165  400
3      -1437  1170  400
4      -1437  1175  400
...      ...   ...  ...
251999  1357  4009  400
252000  1357  4014  400
252001  1357  4019  400
252002  1357  4024  400
252003  1357  4029  400

[252004 rows x 3 columns]


python -m apache_beam.examples.wordcount --input notebooks/npy_metadata_output.csv --output notebooks/counts # type: ignore